# Aula 08 — Parte D: A Reordenação (*Rerank*) no Motor do Grupo (Baixada Santista)
**Projeto Integrador III — Motor de Busca Municipal e Histórico da Baixada Santista**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

**Grupo:** `ivan_gustavo_jorge` · *Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*


In [1]:
# 1. Setup oficial do motor08 e carregamento do corpus congelado (132 paragrafos validos)
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")
source("config.R")

arq_csv <- if (file.exists("estrutura/banco-de-dados/corpus.csv")) {
  "estrutura/banco-de-dados/corpus.csv"
} else {
  "corpus.csv"
}
cp <- read.csv(arq_csv, stringsAsFactors = FALSE, fileEncoding = "latin1")
validos <- !is.na(cp$texto) & (nchar(cp$texto) > cfg$minimo)
cp <- cp[validos, ]
docs <- setNames(cp$texto, cp$id)
origem <- setNames(ifelse(grepl("guaruja", cp$titulo, ignore.case = TRUE), "Guaruja",
                   ifelse(grepl("santos",  cp$titulo, ignore.case = TRUE), "Santos", "Bertioga")),
                   cp$id)
ix <- montar(docs, cfg)
cat("Corpus carregado com sucesso! Total de documentos validos:", length(docs), "\n")
print(table(origem))
estado()


Corpus carregado com sucesso! Total de documentos validos: 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 

[1] "MOTOR_VERSAO: motor08 v2 2026-10-22"


In [2]:
# 2. Carregamento do gabarito e das necessidades de informacao
arq_qrels <- if (file.exists("estrutura/banco-de-dados/qrels_jorge_2026-10-13.csv")) {
  "estrutura/banco-de-dados/qrels_jorge_2026-10-13.csv"
} else {
  "qrels_jorge_2026-10-13.csv"
}
arq_nec <- if (file.exists("estrutura/banco-de-dados/necessidades.csv")) {
  "estrutura/banco-de-dados/necessidades.csv"
} else {
  "necessidades.csv"
}
qrels <- ler_qrels(arq_qrels, juiz = cfg$juizes)
nec   <- read.csv(arq_nec, stringsAsFactors = FALSE, fileEncoding = "UTF-8")
consultas <- setNames(nec$texto_consulta, nec$consulta)
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]
cat("Consultas ativas (R > 0):", paste(usar, collapse = ", "), "\n")


Consultas ativas (R > 0): q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12 
Conjunto dev: q01, q02, q03, q04, q05, q06, q07, q08 
Conjunto teste: q09, q10, q11, q12 


## Módulo 7 — O reordenador no motor do grupo: uma consulta, lendo os textos
Implementação de `proximidade()`, `reordenar()`, `estagio1()` e `rerank_prox()` sobre o estágio 1 (`cfg$modelo = "bm25"`) e análise qualitativa na consulta $q_{02}$ (`"porto de Santos"`).


In [3]:
proximidade <- function(qt, dt, janela = 3) {        # qt: tokens da consulta; dt: tokens do documento
  s <- 0                                             # o escore comeca em zero
  for (i in seq_along(qt)) for (j in seq_along(qt))  # todos os pares de posicoes da consulta...
    if (i < j && qt[i] != qt[j]) {                   # ...com i antes de j e termos diferentes
      pos_i <- which(dt == qt[i])                    # onde o termo i aparece no documento
      pos_j <- which(dt == qt[j])                    # onde o termo j aparece
      if (length(pos_i) && length(pos_j)) {          # os dois aparecem?
        d <- outer(pos_j, pos_i, "-")                # todas as distancias (j menos i)
        d <- d[d > 0 & d <= janela]                  # so j DEPOIS de i, e dentro da janela
        if (length(d)) s <- s + 1 / min(d)           # soma 1 / a menor distancia
      }
    }
  s                                                  # devolve o escore
}
reordenar <- function(rk, k, sc) {               # rk: ranking do estagio 1; sc: escores do estagio 2
  k <- min(k, length(rk))                        # protecao: k nao passa do numero de documentos
  top <- names(rk)[1:k]                          # a cabeca: os k primeiros
  nova <- top[order(-sc[top], seq_along(top))]   # pelo escore, decrescente; empate: ordem do estagio 1
  c(nova, names(rk)[-(1:k)])                     # cabeca reordenada + cauda intacta (NOMES)
}
proximidade(c("a", "b"), c("a", "x", "b"))                         # teste: 0.5
reordenar(c(a = 3, b = 2, c = 1), 2, c(a = 0.1, b = 0.9, c = 5))   # teste: "b" "a" "c"

if (is.null(cfg$modelo)) cfg$modelo <- "bm25"
if (is.null(cfg$lsa_k))  cfg$lsa_k  <- 20
if (is.null(cfg$rrf_c))  cfg$rrf_c  <- 60
if (is.null(cfg$janela)) cfg$janela <- 3
estagio1 <- function(texto, ix, cfg, m) {
  rb <- ranking_bm25(texto, ix, cfg)
  if (cfg$modelo == "bm25") return(rb)
  rl <- ranking_lsa(texto, ix, cfg, m)
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)
}
m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)

rerank_prox <- function(rk, qt, tokens, k, janela) {
  if (k == 0) return(names(rk))
  top <- head(names(rk), k)
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))
  reordenar(rk, k, sc)
}

q  <- "q02"                                                 # consulta de trabalho julgada ('porto de Santos')
rk <- estagio1(consultas[[q]], ix, cfg, m)                  # estagio 1: BM25
qt <- preparar(consultas[[q]], cfg); qt                     # tokens da consulta sem stopwords
round(sapply(ix$tokens[names(rk)[1:10]], function(dt) proximidade(qt, dt, 3)), 3)
rn <- rerank_prox(rk, qt, ix$tokens, 10, 3)
rbind(estagio1 = names(rk)[1:5], rerank = rn[1:5])
docs[rn[1:5][!rn[1:5] %in% names(rk)[1:5]]]                 # subiu para o top-5: doc 58


[1] 0.5
[1] "b" "a" "c"
[1] "porto"  "santos"
87 89 31 88 47 58 57 110 49 53 
 1  1  1  0  1  1  0   0  0  0 
         [,1] [,2] [,3] [,4] [,5]
estagio1 "87" "89" "31" "88" "47"
rerank   "87" "89" "31" "47" "58"


## Módulo 8 — $k$ e janela em todas as consultas (Grade Dev e Relato no Teste)
Avaliação da grade $k \in \{3, 5, 10\} \times w \in \{2, 3, 5\}$ nas 8 consultas de desenvolvimento ($q_{01}..q_{08}$) e relato único nas 4 consultas de teste ($q_{09}..q_{12}$).


In [4]:
ap_rerank <- function(rk, qt, grau, tokens, k, janela, limiar)
  metricas(rerank_prox(rk, qt, tokens, k, janela), grau, limiar)[["AP"]]
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])
rks <- lapply(consultas[usar], function(texto) estagio1(texto, ix, cfg, m))
qts <- lapply(consultas[usar], function(texto) preparar(texto, cfg))
sapply(qts, length)

map0 <- mean(sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 0, 3, cfg$limiar)))
round(map0, 3)
grade <- sapply(c(2, 3, 5), function(w)
  sapply(c(3, 5, 10), function(k)
    mean(sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, k, w, cfg$limiar)))))
rownames(grade) <- c("k3", "k5", "k10"); colnames(grade) <- c("janela2", "janela3", "janela5")
round(grade, 3)

ap0 <- sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 0, 3, cfg$limiar))
ap1 <- sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 10, 3, cfg$limiar))
round(rbind(estagio1 = ap0, rerank = ap1), 3)
dev[ap1 < ap0]  # Perdeu em q04 (0.766 -> 0.754)

k_esc <- 0; j_esc <- 3
tab_teste <- sapply(teste, function(q)
  sapply(c(estagio1 = 0, rerank = k_esc), function(k)
    ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, k, j_esc, cfg$limiar)))
round(tab_teste, 3); round(rowMeans(tab_teste), 3)


q01 q02 q03 q04 q05 q06 q07 q08 q09 q10 q11 q12 
  2   2   2   3   2   3   3   4   4   4   2   3 
[1] 0.744
    janela2 janela3 janela5
k3    0.744   0.744   0.744
k5    0.744   0.744   0.744
k10   0.743   0.743   0.743
           q01   q02   q03   q04   q05  q06   q07   q08
estagio1 0.718 0.791 0.706 0.766 0.735 0.79 0.591 0.854
rerank   0.718 0.791 0.706 0.754 0.735 0.79 0.591 0.854
[1] "q04"
           q09  q10   q11   q12
estagio1 0.843 0.67 0.593 0.767
rerank   0.843 0.67 0.593 0.767
estagio1   rerank 
   0.718    0.718 


## Módulo 9 — A decisão do grupo e o registro em `config.R`
**Decisão:** `rerank_k = 0` e `janela = 3` (não reordenar em produção, mantendo o estágio 1 `bm25`).


In [5]:
source("config.R")
cfg$rerank_k; cfg$janela
ranking_final <- function(texto, ix, cfg, m)
  rerank_prox(estagio1(texto, ix, cfg, m), preparar(texto, cfg),
              ix$tokens, cfg$rerank_k, cfg$janela)
head(ranking_final(consultas[[usar[1]]], ix, cfg, m), 5)


[1] 0
[1] 3
[1] "13"  "55"  "114" "87"  "83" 
